# 02.1 — NumPy vs MATLAB arrays

NumPy is the foundation underneath every other Python ML library. PyTorch tensors are basically NumPy arrays with GPU + autograd glued on. This notebook covers what changes when you go from MATLAB's array model to NumPy's:

* **0-indexing and half-open ranges** (already touched in [00.3](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb); deeper now).
* **Broadcasting** — implicit shape alignment that's both more powerful and more error-prone than MATLAB's.
* **Views vs copies** — a common silent-bug source.
* **Row-major (C-order) vs column-major (Fortran-order)** memory layout.
* **The NumPy-MATLAB conversion table** as a one-stop cheat sheet.

**Prerequisite:** [01.1 syntax basics](../01_python_for_matlab_users/01.1_syntax_basics.ipynb), [00.3 mental model](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb).

## Section 1 — What MATLAB does

In MATLAB, everything is a matrix. A scalar is a 1×1 matrix; a vector is 1×N or N×1; a 2D image is M×N. Arrays:

* Are **1-indexed**: `A(1)` is the first element.
* Use **inclusive ranges**: `A(1:N)` covers all N elements.
* Default to **column-major** memory layout (the way Fortran does it).
* Are **always copied** on assignment: `B = A; B(1) = 99;` does not modify `A`.
* Have **implicit element-wise operators**: `A.*B`, `A./B`, `A.^2`.

Python's NumPy differs on every single one of these. The differences are small but each has bitten every MATLAB user at least once.

## Section 2 — The Python concepts you need

### 2.1 — Array creation

In [ ]:
import numpy as np

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print(a)
print(a.shape, a.dtype)

In [ ]:
# Common constructors
print(np.zeros((3, 4)))       # 3x4 matrix of zeros
print()
print(np.ones((2, 3)))         # 2x3 matrix of ones
print()
print(np.arange(0, 10, 2))     # 0, 2, 4, 6, 8 (half-open range like Python's range())
print()
print(np.linspace(0, 1, 5))    # 5 evenly-spaced points from 0 to 1 INCLUSIVE

**`arange` is half-open; `linspace` is inclusive.** This bites people. Use `linspace` when you want N points across a range; `arange` when you want a known step size.

MATLAB ↔ NumPy creation table:

| MATLAB | NumPy |
|---|---|
| `zeros(3,4)` | `np.zeros((3, 4))` |
| `ones(2,3)` | `np.ones((2, 3))` |
| `eye(5)` | `np.eye(5)` |
| `0:0.1:1` | `np.arange(0, 1.001, 0.1)` (note: half-open!) |
| `linspace(0, 1, 11)` | `np.linspace(0, 1, 11)` |
| `randn(3)` | `np.random.standard_normal((3, 3))` — careful: MATLAB's `randn(3)` means 3×3! |
| `[1 2 3; 4 5 6]` | `np.array([[1, 2, 3], [4, 5, 6]])` |

### 2.2 — Indexing and slicing

Most subtle area. 0-indexed, half-open ranges, negative indices count from the end.

In [ ]:
a = np.arange(10)
print(a)
print(a[0])      # 0 — first element
print(a[-1])     # 9 — last
print(a[2:5])    # [2, 3, 4] — half-open
print(a[:3])     # [0, 1, 2] — start defaults to 0
print(a[7:])     # [7, 8, 9] — end defaults to length
print(a[::2])    # [0, 2, 4, 6, 8] — step of 2
print(a[::-1])   # [9, 8, 7, ...] — reverse

In [ ]:
# 2D indexing — comma-separated axes inside ONE square-bracket
M = np.arange(12).reshape(3, 4)
print(M)
print()
print(M[0, 0])      # top-left scalar (MATLAB: M(1,1))
print(M[1, :])      # row 1 (MATLAB: M(2,:))
print(M[:, 2])      # column 2 (MATLAB: M(:,3))
print(M[1:3, 1:3])  # 2x2 sub-block (MATLAB: M(2:3, 2:3))

**The single-bracket vs MATLAB's parentheses is the most-typed mistake.** MATLAB writes `M(1, 2)`; NumPy writes `M[1, 2]`. Round vs square parens.

### 2.3 — Broadcasting

This is the big one. NumPy lets you operate on arrays of different shapes if they can be *broadcast* to a common shape. The rules:

1. Compare shapes element-by-element from the **right**.
2. Two dimensions are compatible if they're **equal** OR one of them is **1**.
3. If one shape is shorter, pretend it has leading `1`s.

Visual diagram below — the most common pattern is adding a per-row or per-column vector to a matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Setup: a 4x5 matrix M and a length-5 row vector v
M = np.arange(20).reshape(4, 5).astype(float)
v = np.array([10, 20, 30, 40, 50], dtype=float)
result = M + v

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, data, title in zip(
    axes,
    [M, np.broadcast_to(v, M.shape), result],
    [f"M\nshape ({M.shape[0]}, {M.shape[1]})",
     f"v\nshape ({v.shape[0]},) broadcasts to ({M.shape[0]}, {M.shape[1]})",
     f"M + v\nshape ({result.shape[0]}, {result.shape[1]})"],
):
    im = ax.imshow(data, cmap="viridis", vmin=0, vmax=70)
    for (i, j), val in np.ndenumerate(data):
        ax.text(j, i, f"{int(val):d}", ha="center", va="center",
                color="white" if val < 40 else "black", fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()
print("M.shape =", M.shape, "v.shape =", v.shape, "→ result.shape =", result.shape)

The vector `v` is *stretched* (conceptually — NumPy doesn't actually copy memory) to the matrix shape, then added element-wise. The `(5,)` aligns to the trailing axis of `(4, 5)`.

For a **column** broadcast instead, give `v` shape `(4, 1)`:

In [ ]:
u = np.array([100, 200, 300, 400], dtype=float).reshape(4, 1)
print("u shape:", u.shape)
print("M + u =")
print(M + u)

### 2.4 — When broadcasting fails

If shapes don't satisfy the rule, NumPy raises `ValueError: operands could not be broadcast together with shapes ...`. The fix is almost always to **add a length-1 axis** at the right place using `np.newaxis` (alias `None`).

In [ ]:
# Try to add a length-4 vector to a 4x5 matrix WITHOUT specifying axis — fails
u = np.array([100, 200, 300, 400], dtype=float)
try:
    M + u
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Fix: reshape u to (4, 1) so it aligns with the leading axis
u_col = u[:, np.newaxis]
print("u_col shape:", u_col.shape)
print(M + u_col)

**Memorize the `[:, np.newaxis]` and `[np.newaxis, :]` patterns** — they're how you steer broadcasting. The first adds a column axis; the second adds a row axis.

In our codebase: `data/dataset.py` adds a per-class signal to every window of a trial with `class_signals[d][cls][None, ...]` — the added leading axis lets the `(T, A, C)` signal broadcast across the trial's `W` axis. Same operation as `[np.newaxis, ...]` — `None` is shorter.

### 2.5 — Views vs copies

Slicing typically returns a **view** — a new array that shares memory with the original. Modifying the view modifies the original. This is the opposite of MATLAB.

In [ ]:
a = np.arange(10)
b = a[2:6]       # view, NOT copy
b[0] = 999
print("a:", a)
print("b:", b)
# a was modified at index 2!

In [ ]:
# To get a copy, ask for one explicitly
a = np.arange(10)
b = a[2:6].copy()
b[0] = 999
print("a:", a)        # unchanged
print("b:", b)

**Rule of thumb:**

| Operation | Returns |
|---|---|
| Basic slicing `a[1:5]`, `a[:, 0]`, etc. | **view** (shared memory) |
| Fancy indexing `a[[1, 3, 5]]` or boolean `a[a > 0]` | **copy** (new memory) |
| `a.copy()` | always a copy |
| `np.asarray(a)` | view if `a` is already ndarray |
| `np.array(a)` | always a copy |

In our codebase: `MatFileTrialDataset.__getitem__` calls `torch.from_numpy(np.ascontiguousarray(features_np))` — `torch.from_numpy` cannot wrap arrays with exotic memory layouts (e.g. the negative strides a `[::-1]` reversal produces), so `ascontiguousarray` guarantees a clean, contiguous buffer first.

### 2.6 — Shape manipulation

The big four: `reshape`, `transpose`, `expand_dims`, `squeeze`.

In [ ]:
a = np.arange(24)
print("flat:", a.shape)

b = a.reshape(2, 3, 4)
print("reshape 2x3x4:", b.shape)

c = b.transpose(2, 0, 1)
print("transpose to (axis2, axis0, axis1):", c.shape)

d = np.expand_dims(c, axis=0)      # same as c[None, ...]
print("add leading axis:", d.shape)

e = d.squeeze(0)
print("squeeze axis 0:", e.shape)

**MATLAB ↔ NumPy shape ops:**

| MATLAB | NumPy |
|---|---|
| `reshape(A, [2 3 4])` | `A.reshape(2, 3, 4)` |
| `A.'` (transpose; note `A'` is the complex-conjugate transpose) | `A.T` (2D) or `A.transpose(...)` (N-D) |
| `permute(A, [3 1 2])` | `A.transpose(2, 0, 1)` — note 0-indexed! |
| `A(:,:,1)` | `A[:, :, 0]` |
| `squeeze(A)` | `A.squeeze()` |

**Beware MATLAB's permute order:** `permute(A, [3 1 2])` in MATLAB means "new axis 1 = old axis 3" using 1-indexing. NumPy's `A.transpose(2, 0, 1)` means the same thing but 0-indexed. Off-by-one bugs here are common when porting.

## Section 3 — The `neural_data_decoding` implementation

Look at `_window_trial` in `data/mat_dataset.py` — pure NumPy, lots of axis manipulation:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/mat_dataset.py").read_text().splitlines()
start = next((i for i, line in enumerate(src) if line.startswith("def _window_trial")), -1)
for i in range(start, min(start + 30, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Things to spot:

* `np.empty((nc, dw, na, nw), dtype=data.dtype)` — pre-allocate the output array.
* Loop with `enumerate(starts)` (the indexed iteration pattern from [01.2](../01_python_for_matlab_users/01.2_control_flow.ipynb)).
* `data[:, int(s) : int(s) + data_width, :]` — basic slicing returns a view; no copy until the assignment writes it.
* `np.transpose(windowed, (3, 1, 2, 0))` — `(C, T, A, W) → (W, T, A, C)` per the codebase's convention (covered in [02.2](02.2_array_axis_conventions.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — broadcasting

Given a `(10, 4)` matrix `X` and a length-4 vector `mu`, subtract `mu` from each row of `X`. Two ways: with explicit broadcasting, then with `np.broadcast_to` to confirm the shapes.

In [ ]:
X = np.random.standard_normal((10, 4))
mu = X.mean(axis=0)
print("X.shape =", X.shape, "mu.shape =", mu.shape)
# Your attempt here


In [ ]:
# Reveal:
centered = X - mu
print("centered.shape =", centered.shape)
print("per-column mean after centering:", centered.mean(axis=0))   # ≈ 0

### Exercise 4.2 — view vs copy

In [ ]:
a = np.arange(20).reshape(4, 5)
sub = a[1:3, 1:4]
sub[0, 0] = -1
# Does a have a -1 in it now? Predict, then run.
print(a)

## Section 5 — Common errors

### `ValueError: operands could not be broadcast together with shapes (4,) (5,)`

Vector shapes don't line up. Reshape one to add a length-1 axis where needed: `u[:, np.newaxis]` for column-style, `u[np.newaxis, :]` for row-style.

### `IndexError: index N is out of bounds for axis 0 with size N`

You used a 1-indexed value where Python wants 0-indexed. Subtract 1, or use `range(N)` instead of `range(1, N+1)`.

### Mysterious mutation: "I didn't touch `a` but it changed!"

A slice is a view. Either work on `a.copy()`, or be explicit about what shares memory.

### `ValueError: cannot reshape array of size N into shape (...)`

Reshape requires the total element count to match. `np.arange(10).reshape(3, 4)` fails because 10 ≠ 12. Use `-1` for one axis to let NumPy infer: `arr.reshape(-1, 4)` → "whatever rows, 4 columns."

### `IndexError: only integers, slices (\`:\`), ellipsis ... are valid indices`

You indexed with a float: `arr[3.0]` fails; `arr[3]` works. Convert with `int(...)` if needed. A related `TypeError: only integer scalar arrays can be converted to a scalar index` appears when you pass a multi-element array where a single index is expected.

## Section 6 — Further reading

- [NumPy docs — basic indexing](https://numpy.org/doc/stable/user/basics.indexing.html). The single best reference.
- [NumPy docs — broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html). Worked-through examples.
- [NumPy for MATLAB users](https://numpy.org/doc/stable/user/numpy-for-matlab-users.html) — the official MATLAB-to-NumPy translation guide.
- [Views and copies](https://numpy.org/doc/stable/user/basics.copies.html) — exactly when each happens.

Next up: **[02.2 array axis conventions](02.2_array_axis_conventions.ipynb)** — how MATLAB's `SSCTB`, PyTorch's `(N, C, H, W)`, and the codebase's `(W, T, A, C)` relate to each other.